# autoprof

> Functionality for running autoprof to measure the ICL in images of galaxy clusters.

In [ ]:
# | default_exp autoprof

In [ ]:
import glob
import numpy as np
from astropy.io import fits

import importlib.util
import sys
import os
import traceback
from pathlib import Path
from textwrap import dedent

import autoprof
from autoprof import Pipeline 


In [ ]:
# | hide
# Additional imports used in the examples


In [ ]:
# | export

def clean_images_for_AP(image_pattern, output_directory=None, nan_value_replace='extreme'):
    """
    Cleans all images matching the given pattern by replacing NaNs with 99 or nan min of the image.
    """
    image_files = sorted(glob.glob(image_pattern))
    
    if not image_files:
        print(f"No files matched pattern: {image_pattern}")
        return

    for image in image_files:
        image_path = Path(image)
        print(f"\nProcessing Image: {image_path.name}")
        new_clean_image_name = f'cleaned_{image_path.name}'

        with fits.open(image_path) as hdul:
            image_data = hdul[0].data.copy()
            image_header = hdul[0].header.copy()
            ny, nx = image_data.shape
            nan_mask = np.isnan(image_data)
            print(f"Number of NaN values: {nan_mask.sum()}")

            clean_image_data = image_data.copy()
            if nan_value_replace == 'extreme':   
                print("replacing nans to 99")
                clean_image_data[nan_mask] = 99
            elif nan_value_replace == 'nanmin':
                print("replacing nans to nanmin")
                clean_image_data[nan_mask] = np.nanmin(clean_image_data)

        
        output_dir = Path(output_directory) if output_directory else image_path.parent
        clean_image_path = output_dir / new_clean_image_name

        hdu = fits.PrimaryHDU(data=clean_image_data, header=image_header)
        hdu.writeto(clean_image_path, overwrite=True)
        print(f"Cleaned image saved to: {clean_image_path}")

def cleaning_process_AP(imdir,mode="bulk", clusterid=None, bands=None, nan_value_replace='extreme'):
    boxsize = 300
    if bands is None:
        bands = ["H", "J", "Y", "HJY"]

    for band in bands:
        if mode == "single":
            if not clusterid:
                raise ValueError("clusterid must be provided in single mode.")
            pattern = f"{imdir}{clusterid}_{band}_BKGSUB_boxsize_{boxsize}.fits"
        
        elif mode == "bulk":
            pattern = f"{imdir}1*_{band}_BKGSUB_boxsize_{boxsize}.fits"
       
        else:
            raise ValueError("mode must be either 'bulk' or 'single'")
        
        clean_images_for_AP(image_pattern=pattern, nan_value_replace=nan_value_replace)
        

In [ ]:

def run_autoprof(
    ids,
    image_files,
    mask_files,
    mode = "image list",
    unit_type= "intensity",
    gscale=0.4,
    pixelscale=0.3,
    zeropoint = 23.9,
    out_dir="./",
    config_name="basic_config.py",
    log_path="AutoProf.log",
    fourier_orders = None,
):
    """
    Running AutoProf on given image and mask files with a custom configuration.
    
    Parameters:
    - ids: List of strings for each image
    - image_files: List of paths to the image fits files
    - mask_files: List of paths to the mask fits files
    - mode : "image list" then ids, image and mask files needs to be arrays so it ca run for bulk.
    - unit_type: Either 'intensity' or 'mag'
    - gscale: Geometric scaling factor, default for noise diagnostics is 0.4 ~0.15dex, for clusters 0.1~0.05dex
    - out_dir: Output directory for saving results and logs
    - config_name: Name for the config Python file, so far I only magaed to run this software by calling its "basic_config" ...
      it crashes once this naming change, since it tries to access certain modules associated with it.

    - fourier_orders : To turn on extraction of Fourier coefficients, useful for lopsidedness analysis. I made it optional. Takes in a tuple (1,2,3)
    """
    os.makedirs(out_dir, exist_ok=True)
    
    config_file = os.path.join(out_dir, config_name)
    
    # Write the config file for AP
    with open(config_file, "w") as f:
        f.write(dedent(f"""\
import numpy as np
ap_process_mode = f"{mode}"
ap_name = {ids}
ap_image_file = {image_files}
ap_mask_file = {mask_files}
ap_saveto = "{out_dir}"
ap_pixscale = {pixelscale}
ap_zeropoint = {zeropoint}
ap_samplegeometricscale = {gscale}
ap_doplot = True
ap_extractfull = True
ap_fluxunits = "{unit_type}"
ap_isoclip = True
ap_isoclip_nsigma = 5
ap_ellipsefit = True
ap_fix_pa = False
ap_initial_pa = 45.0 
ap_fix_ellipticity = False
ap_initial_ellipticity = 0.3 
{"ap_iso_measurecoefs = " + str(fourier_orders) if fourier_orders else ""}
ap_new_pipeline_steps = [
    "mask segmentation map",
    "background",
    "psf",
    "center",
    "isophoteinit",
    "isophotefit",
    "isophoteextract",
    "writeprof",
]
"""))
    
    os.chdir(out_dir)   # Need to change the working directory for outputs, diagnostic plots will not be saved properly if this is not applied 


    try:
        import sys
        if config_name.replace(".py", "") in sys.modules:
            del sys.modules[config_name.replace(".py", "")]
        
        PIPELINE = Pipeline.Isophote_Pipeline(loggername=log_path)
        PIPELINE.Process_ConfigFile(config_name.replace(".py", ""))

        print(f"AutoProf completed successfully for {ids}!")
    except Exception as e:
        print(f"AutoProf failed for {ids}. Logging error...")

        error_log_file = Path(out_dir) / f"autoprof_error_{ids[0]}.log"
        with open(error_log_file, "w") as log:
            log.write(f"AutoProf failed for {ids}\n")
            log.write(f"Error type: {type(e).__name__}\n")
            log.write(f"Error message: {str(e)}\n")
            log.write("Full traceback:\n")
            log.write(traceback.format_exc())

        print(f"Error log saved to: {error_log_file}")


## Examples

In [ ]:
ids = ["1eRASS_J035423.6-475145"]
imf = ['/Users/tutkukolcu/Documents/Notticl/Autoprof_Tests/Data_diagnostics/V5/Steven/Clusters/cleaned_1eRASS_J035423.6-475145_H_BKGSUB_boxsize_300.fits']
maskf = ['/Users/tutkukolcu/Documents/Notticl/Autoprof_Tests/Data_diagnostics/V5/Steven/Clusters/mask_1eRASS_J035423.6-475145_HJY_4000kpc_CAL_rms_2_default.fits']
outdir = '/Users/tutkukolcu/Documents/Notticl/Autoprof_Tests/Data_diagnostics/V5/Steven/Clusters/'

# fourier_orders = (1,)
fourier_orders = None
run_autoprof(ids=ids, image_files=imf, mask_files=maskf, out_dir=outdir, fourier_orders = fourier_orders)

In [ ]:
# Running for multiple clusters

clusters = ['1eRASS_J035423.6-475145', '1eRASS_J035756.9-490001', '1eRASS_J040558.4-491553', '1eRASS_J041723.0-474844']

bands = ["H", "J", "Y", "HJY"]

unit_types = ['intensity']
dtype = 'BKGSUB'
boxsize = 300
rad = 4000
gscale = 0.4

im_dir = '/Users/tutkukolcu/Documents/Notticl/Autoprof_Tests/Data_diagnostics/V5/Steven/Clusters/'
out_dir = '/Users/tutkukolcu/Documents/Notticl/Autoprof_Tests/Data_diagnostics/V5/Steven/Profiles_BKGSUB/'

# cleaning_process_AP(im_dir, mode='bulk')
# cleaning_process_AP(imdir=im_dir, clusterid=clusters[0], mode='single', bands = ["H", "J", "Y", "HJY"], nan_value_replace='extreme')

print("Starting Autoprof Processing...")

for clusterid in clusters:
    for band in bands:
        for unit in unit_types:
            image_files = sorted(glob.glob(im_dir + f'cleaned_{clusterid}_{band}_{dtype}_boxsize_{boxsize}.fits'))
            mask_files  = sorted(glob.glob(im_dir + f'mask_{clusterid}_HJY_{rad}kpc_CAL_rms_2_default.fits'))

            ids = [
                Path(image).name.removeprefix("cleaned_").split('.fits')[0]
                + f'_{unit}_gscale_{gscale}_'
                for image in image_files
            ]
            
            fourier_orders = (1,)
            
            run_autoprof(
                ids=ids,
                image_files=image_files,
                mask_files=mask_files,
                unit_type=unit,
                gscale=gscale,
                out_dir=out_dir,
                fourier_orders = fourier_orders
            )

print("All done!")
